<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook inventories variant counts for Phase 1 long-read and short-read callsets. It counts SVTYPEs from joint SV VCFs (including a 50-sample ONT Sniffles set) and uses Hail `sample_qc` and `count_rows` for SNVs and indels in the 1,027-sample and ~990-sample overlaps.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json

In [ ]:
import pysam
from pysam import VariantFile

from collections import defaultdict
from collections import Counter

from tqdm.notebook import tqdm

In [ ]:
from google.cloud import storage

In [ ]:
import hail as hl
from hail.plot import show
from pprint import pprint

## Define helper functions

In [ ]:
# True if a Hail MatrixTable already exists at gcs_path (checks for its README.txt).
def mt_exists(gcs_path):
    (gcs_bucket_name, gcs_obj) = re.split("\/", re.sub("gs://", "", gcs_path), maxsplit=1)
    
    storage_client = storage.Client()
    gcs_bucket = storage_client.bucket(gcs_bucket_name)
    stats = storage.Blob(bucket=gcs_bucket, name=f'{gcs_obj}/README.txt').exists(storage_client)
    
    return stats

In [ ]:
# True if a Hail VariantDataset already exists at gcs_path (checks its reference_data README.txt).
def vds_exists(gcs_path):
    (gcs_bucket_name, gcs_obj) = re.split("\/", re.sub("gs://", "", gcs_path), maxsplit=1)
    
    storage_client = storage.Client()
    gcs_bucket = storage_client.bucket(gcs_bucket_name)
    stats = storage.Blob(bucket=gcs_bucket, name=f'{gcs_obj}/reference_data/README.txt').exists(storage_client)
    
    return stats

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']
workspace = os.environ['WORKSPACE_NAME']
namespace = os.environ['WORKSPACE_NAMESPACE']

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Short-read SV callset (GATK-SV v7, subset to long-read samples), downloaded once, and its sample IDs.
if not os.path.exists("cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz"):
    !gsutil cp {AOU_LR_DATA}/yulia/cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz .

In [ ]:
sr_sv_samples = !zgrep -m1 '^#CHROM' cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(sr_sv_samples)

In [ ]:
# Long-read joint SV callsets at two calibration sensitivities (sens_09 and the stricter sens_07).
if not os.path.exists("concat_annotated.sens_09.vcf.gz"):
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_09.vcf.gz .
        
if not os.path.exists("concat_annotated.sens_07.vcf.gz"):        
    !gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_07.vcf.gz .

In [ ]:
sv_sens_09_vcf = 'concat_annotated.sens_09.vcf.gz'
sv_sens_07_vcf = 'concat_annotated.sens_07.vcf.gz'

In [ ]:
# Peek at the first records of the sens_09 callset.
!cat {sv_sens_09_vcf} | zcat | head -n 2000 | grep -v '^#' | head -n 3 | cut -f1-9

In [ ]:
sv_sens_09_in = VariantFile(sv_sens_09_vcf)  # auto-detect input format

for i, rec in enumerate(sv_sens_09_in):
    print(f'{i} {rec.chrom} {rec.pos} {rec.info.values()}')
    
    if i > 10:
        break

In [ ]:
def count_samples_in_vcf(vcf_file_path):
    """
    Counts the number of samples in a VCF file.

    Parameters:
        vcf_file_path (str): Path to the VCF file.
    
    Returns:
        int: Number of samples in the VCF.
    """
    try:
        # Open the VCF file
        vcf = pysam.VariantFile(vcf_file_path)
        
        # Get the sample names
        sample_names = list(vcf.header.samples)
        
        # Return the number of samples
        return len(sample_names)
    except FileNotFoundError:
        raise FileNotFoundError(f"VCF file not found at: {vcf_file_path}")

In [ ]:
# Number of samples, and number of SVs by SVTYPE, in each long-read callset.
count_samples_in_vcf(sv_sens_09_vcf)

In [ ]:
def count_variants_by_svtype(vcf_file_path, field='SVTYPE'):
    """
    Opens a VCF file and counts the number of variants by SVTYPE in the INFO field.
    
    Parameters:
        vcf_file_path (str): Path to the VCF file.
    
    Returns:
        dict: A dictionary where keys are SVTYPE values and values are counts.
    """
    # Open the VCF file
    try:
        vcf = pysam.VariantFile(vcf_file_path)
    except FileNotFoundError:
        raise FileNotFoundError(f"VCF file not found at: {vcf_file_path}")
    
    # Counter for SVTYPE occurrences
    svtype_counts = Counter()
    
    # Iterate through each record in the VCF
    for record in vcf.fetch():
        # Access the INFO field and get SVTYPE, if available
        svtype = record.info.get(field, None)
        if svtype:
            svtype_counts[svtype] += 1
    
    return dict(svtype_counts)

In [ ]:
count_variants_by_svtype(sv_sens_09_vcf)

In [ ]:
count_variants_by_svtype(sv_sens_07_vcf)

In [ ]:
!ls *.vcf.gz

In [ ]:
# Count, for each sample, the SVs of each SVTYPE where the sample carries a non-reference allele
# (optionally PASS sites only).
def count_svtype_per_sample(vcf_file_path, field='SVTYPE', passonly=False):
    """
    Iterates over each line in a VCF file and counts the number of variants
    of each SVTYPE for each sample.

    Parameters:
        vcf_file_path (str): Path to the VCF file.

    Returns:
        dict: A nested dictionary where the outer keys are sample names,
              inner keys are SVTYPEs, and values are counts.
    """
    # Open the VCF file
    try:
        vcf = pysam.VariantFile(vcf_file_path)
    except FileNotFoundError:
        raise FileNotFoundError(f"VCF file not found at: {vcf_file_path}")
        
    num_records_line = !zgrep -vc ^'#' {vcf_file_path}
    num_records = int(num_records_line[0])
    
    # Initialize the dictionary for counts
    svtype_counts = defaultdict(lambda: defaultdict(int))
    
    # Iterate through each record in the VCF
    for record in tqdm(vcf.fetch(), total=num_records, desc="Processing VCF records"):
        # Skip non-PASS variants
        if passonly and list(record.filter.keys()) != ['PASS']:
            continue

        # Get the SVTYPE from the INFO field
        svtype = record.info.get(field, None)
        if not svtype:
            continue  # Skip if SVTYPE is not present
            
        svtype = svtype[0] if type(svtype) is tuple else svtype
        
        # Update counts for each sample
        for sample in record.samples:
            genotype = record.samples[sample].get('GT')
            if genotype is None or any(gt is None for gt in genotype):
                continue  # Skip no-calls (./.)
            
            if any(gt != 0 for gt in genotype):  # Check for non-reference alleles
                svtype_counts[sample][svtype] += 1
                
    return svtype_counts

In [ ]:
# Per-sample SV counts for each long-read callset, cached as JSON and copied to the workspace bucket.
if not os.path.exists("sv_sens_09_counts.json"):
    sv_sens_09_counts = count_svtype_per_sample(sv_sens_09_vcf)

    # Write to a JSON file
    with open('sv_sens_09_counts.json', 'w') as json_file:
        json.dump(sv_sens_09_counts, json_file, indent=4)

with open('sv_sens_09_counts.json', 'r') as json_file:
    sv_sens_09_counts = json.load(json_file)

In [ ]:
sv_sens_09_counts

In [ ]:
if not os.path.exists("sv_sens_07_counts.json"):
    sv_sens_07_counts = count_svtype_per_sample(sv_sens_07_vcf)

    # Write to a JSON file
    with open('sv_sens_07_counts.json', 'w') as json_file:
        json.dump(sv_sens_07_counts, json_file, indent=4)

with open('sv_sens_07_counts.json', 'r') as json_file:
    sv_sens_07_counts = json.load(json_file)

In [ ]:
sv_sens_07_counts

In [ ]:
with open('sv_sens_07_counts.json', 'w') as json_file:
    json.dump(sv_sens_07_counts, json_file, indent=4) 

In [ ]:
!gsutil cp sv_sens_0*_counts.json $WORKSPACE_BUCKET/scratch/

## List long read samples

In [ ]:
# Sample IDs in the long-read SV callset (1,074 samples, including HPRC), also written to samples_1074.txt.
lr_sv_samples = !zgrep -m1 '^#CHROM' concat_annotated.sens_09.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(lr_sv_samples)

In [ ]:
!zgrep -m1 '^#CHROM' concat_annotated.sens_09.vcf.gz | cut -f10- | sed 's/\t/\n/g' > samples_1074.txt


In [ ]:
with open('samples_1074.txt', 'r') as file:
    sample_names = file.readlines()

sample_names = [name.strip() for name in sample_names]

In [ ]:
len(sample_names)

## List long read samples without HPRC samples

In [ ]:
# Drop the HPRC samples (IDs starting HG or NA), leaving the 1,027 AoU long-read samples.
common_samples_1027 = [element for element in lr_sv_samples if not (element.startswith('HG') or element.startswith('NA'))]
len(common_samples_1027)

## List long read samples with GATK-SV calls available

In [ ]:
# AoU long-read samples that also have short-read SV calls (990), written one per line for bcftools.
common_samples_990 = list(set(sr_sv_samples) & set(lr_sv_samples))
len(common_samples_990)

In [ ]:
with open('samples_990.txt', 'w') as file:
    for sample_name in common_samples_990:
        file.write(f'{sample_name}\n')

In [ ]:
# Build bcftools 1.21, then subset both long-read callsets to the 990 samples.
!wget -O bcftools-1.21.tar.bz2 https://github.com/samtools/bcftools/releases/download/1.21/bcftools-1.21.tar.bz2

In [ ]:
!bunzip2 -c bcftools-1.21.tar.bz2 | tar xvf -

In [ ]:
!cd bcftools-1.21 && make

In [ ]:
!./bcftools-1.21/bcftools

In [ ]:
!./bcftools-1.21/bcftools view -S samples_990.txt --force-samples -o concat_annotated.sens_07.subset.vcf.gz -O z concat_annotated.sens_07.vcf.gz


In [ ]:
!./bcftools-1.21/bcftools view -S samples_990.txt --force-samples -o concat_annotated.sens_09.subset.vcf.gz -O z concat_annotated.sens_09.vcf.gz


In [ ]:
#!./bcftools-1.21/bcftools view -S samples_990.txt --force-samples -o concat_annotated.sens_09.subset.vcf.gz -O z concat_annotated.sens_09.vcf.gz


In [ ]:
!ls -lh *.vcf.gz

In [ ]:
# Per-sample SV counts in the short-read callset (PASS sites only), cached as JSON.
sv_sr_vcf = 'cohort_AoUSVPhaseII.v7.LRsamples.vcf.gz'

In [ ]:
if not os.path.exists("sv_sr_counts.json"):
    sv_sr_counts = count_svtype_per_sample(sv_sr_vcf, passonly=True)

    # Write to a JSON file
    with open('sv_sr_counts.json', 'w') as json_file:
        json.dump(sv_sr_counts, json_file, indent=4)

with open('sv_sr_counts.json', 'r') as json_file:
    sv_sr_counts = json.load(json_file)

In [ ]:
sv_sr_counts

In [ ]:
!gsutil cp sv_sr_counts.json $WORKSPACE_BUCKET/scratch/

## Download Sniffles2 calls for HPRC samples

In [ ]:
# Switch gcloud to a second account and log in.
!gcloud config configurations activate second-user
!gcloud auth login --brief

In [ ]:
# Sniffles2 SV calls for HPRC sample HG002.
!gsutil ls -lh {AOU_LR_DATA}/HG002_filtered.vcf.gz
    

## Download Sniffles2 calls for 50 ONT samples

In [ ]:
# Per-sample SV counts from the Sniffles2 calls of the ONT samples in the long-read cohort, cached as JSON.
sniffles_vcfs = !gsutil ls {AOU_LR_DATA}/ONT/outputs/GRCh38/variants/sv/*.sniffles.vcf.gz

In [ ]:
sv_ont_counts = {}

for sniffles_vcf in sniffles_vcfs:
    bn = re.sub(".sniffles.vcf.gz", "", os.path.basename(sniffles_vcf))
    
    if bn in common_samples_1027:
        !gsutil cp {sniffles_vcf} .
        sv_ont_sample_counts = count_svtype_per_sample(f'{bn}.sniffles.vcf.gz')
        sv_ont_counts[bn] = sv_ont_sample_counts[bn]

sv_ont_counts

In [ ]:
if not os.path.exists("sv_ont_counts.json"):
    # Write to a JSON file
    with open('sv_ont_counts.json', 'w') as json_file:
        json.dump(sv_ont_counts, json_file, indent=4)

with open('sv_ont_counts.json', 'r') as json_file:
    sv_ont_counts = json.load(json_file)

In [ ]:
!gsutil cp sv_ont_counts.json $WORKSPACE_BUCKET/scratch/

In [ ]:
sv_ont_counts

## Initialize Hail

In [ ]:
# Start Hail, using GRCh38 coordinates.
#spark_conf_more_ram = dict()
#spark_conf_more_ram["spark.executor.memory"] = "8g"
#spark_conf_more_ram["spark.driver.memory"] = "196g"

#hl.init(idempotent=True, spark_conf=spark_conf_more_ram)

hl.init(idempotent=True)

In [ ]:
hl.default_reference('GRCh38')

## Load Hail MT subset to samples with long reads

In [ ]:
# Short-read SNV/indel MatrixTable subset to the 1,027 long-read samples (made by 00_init_subset_vds):
# per-sample QC, Ti/Tv mean and standard deviation, and counts of SNVs, insertions, and deletions.
mt_1027 = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.1027.mt')

In [ ]:
mt_qc_1027 = hl.sample_qc(mt_1027)

In [ ]:
mt_qc_1027.describe()

In [ ]:
mt_qc_1027.cols().show()

In [ ]:
mt_qc_1027.aggregate_cols(hl.agg.stats(mt_qc_1027.sample_qc.r_ti_tv))

In [ ]:
stats = mt_qc_1027.aggregate_cols(hl.agg.stats(mt_qc_1027.sample_qc.r_ti_tv))
print(f"Mean r_ti_tv: {stats['mean']:.4f}")
print(f"Standard deviation: {stats['stdev']:.4f}")

In [ ]:
snv_count_1027 = mt_1027.filter_rows(hl.is_snp(mt_1027.alleles[0], mt_1027.alleles[1])).count_rows()
insertion_count_1027 = mt_1027.filter_rows(hl.is_insertion(mt_1027.alleles[0], mt_1027.alleles[1])).count_rows()
deletion_count_1027 = mt_1027.filter_rows(hl.is_deletion(mt_1027.alleles[0], mt_1027.alleles[1])).count_rows()

In [ ]:
mt_qc_1027.cols()

In [ ]:
print(f'# SNV: {snv_count_1027}')
print(f'# INS: {insertion_count_1027}')
print(f'# DEL: {deletion_count_1027}')

## Load Hail MT subset to samples with long reads with GATK-SV calls

In [ ]:
# Same for the 990 samples that also have short-read SV calls.
mt_990 = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.990.mt')

In [ ]:
mt_qc_990 = hl.sample_qc(mt_990)

In [ ]:
mt_qc_990.describe()

In [ ]:
mt_qc_990.cols().show()

In [ ]:
snv_count_990 = mt_990.filter_rows(hl.is_snp(mt_990.alleles[0], mt_990.alleles[1])).count_rows()
insertion_count_990 = mt_990.filter_rows(hl.is_insertion(mt_990.alleles[0], mt_990.alleles[1])).count_rows()
deletion_count_990 = mt_990.filter_rows(hl.is_deletion(mt_990.alleles[0], mt_990.alleles[1])).count_rows()

In [ ]:
print(f'# SNV: {snv_count_990}')
print(f'# INS: {insertion_count_990}')
print(f'# DEL: {deletion_count_990}')

In [ ]:
mt_qc_990 = hl.sample_qc(mt_990)

In [ ]:
mt_qc_990.describe()

In [ ]:
mt_qc_990.aggregate_cols(hl.agg.stats(mt_qc_990.sample_qc.r_ti_tv))

In [ ]:
mt_qc_990.aggregate_cols(hl.agg.stats(mt_qc_990.sample_qc.r_het_hom_var))

## Load Hail MT on 1027 long read samples

In [ ]:
# Long-read joint-call MatrixTable for the 1,027 samples: per-sample QC, and counts of SNVs, insertions,
# and deletions.
lr_mt = hl.read_matrix_table(f'{AOU_LR_DATA}/outputs/T2T/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1027/cohort_for_GLNexus_2023Q1_1027.mt')

In [ ]:
lr_qc_mt = hl.sample_qc(lr_mt)

In [ ]:
lr_qc_mt.describe()

In [ ]:
lr_qc_mt.cols().show()

In [ ]:
snv_count_lr = lr_mt.filter_rows(hl.is_snp(lr_mt.alleles[0], lr_mt.alleles[1])).count_rows()
insertion_count_lr = lr_mt.filter_rows(hl.is_insertion(lr_mt.alleles[0], lr_mt.alleles[1])).count_rows()
deletion_count_lr = lr_mt.filter_rows(hl.is_deletion(lr_mt.alleles[0], lr_mt.alleles[1])).count_rows()

In [ ]:
print(f'# SNV: {snv_count_lr}')
print(f'# DEL: {deletion_count_lr}')
print(f'# INS: {insertion_count_lr}')

In [ ]:
lr_qc_mt.aggregate_cols(hl.agg.stats(lr_qc_mt.sample_qc.r_ti_tv))

In [ ]:
lr_qc_mt.aggregate_cols(hl.agg.stats(lr_qc_mt.sample_qc.r_het_hom_var))

## Subset long read Hail MT to samples with GATK-SV calls

In [ ]:
# Subset the long-read MatrixTable to the 990 samples, keeping sites where at least one of them carries a
# non-reference allele, and repeat the counts. ('or True' forces a rewrite.)
if not mt_exists(f'{bucket}/scratch/lrs-subset.990.mt') or True:
    lr_samples_990 = hl.literal(set(common_samples_990))
    lr_subset_mt = lr_mt.filter_cols(lr_samples_990.contains(lr_mt.s))
    lr_subset_mt = lr_subset_mt.filter_rows(hl.agg.any(lr_subset_mt.GT.is_non_ref()))

    lr_subset_mt.write(f'{bucket}/scratch/lrs-subset.990.mt', overwrite=True)
    
lr_subset_mt = hl.read_matrix_table(f'{bucket}/scratch/lrs-subset.990.mt')

In [ ]:
len(lr_subset_mt.s.collect())

In [ ]:
lr_subset_qc_mt = hl.sample_qc(lr_subset_mt)

In [ ]:
lr_subset_qc_mt.describe()

In [ ]:
lr_subset_qc_mt.cols().show()

In [ ]:
snv_count_lr_subset = lr_subset_mt.filter_rows(hl.is_snp(lr_subset_mt.alleles[0], lr_subset_mt.alleles[1])).count_rows()
insertion_count_lr_subset = lr_subset_mt.filter_rows(hl.is_insertion(lr_subset_mt.alleles[0], lr_subset_mt.alleles[1])).count_rows()
deletion_count_lr_subset = lr_subset_mt.filter_rows(hl.is_deletion(lr_subset_mt.alleles[0], lr_subset_mt.alleles[1])).count_rows()

In [ ]:
print(f'# SNV: {snv_count_lr_subset}')
print(f'# DEL: {deletion_count_lr_subset}')
print(f'# INS: {insertion_count_lr_subset}')

In [ ]:
lr_subset_qc_mt.aggregate_cols(hl.agg.stats(lr_subset_qc_mt.sample_qc.r_ti_tv))

In [ ]:
lr_subset_qc_mt.aggregate_cols(hl.agg.stats(lr_subset_qc_mt.sample_qc.r_het_hom_var))